In [ ]:
########=======MONTHLY RANDOM FOREST BIAS CORRECTION FOR SRPs========#############
##================Replace CHIRPS with other SRPs, e.g., MSWEP, TAMSAT, etc.============####

import pandas as pd
import numpy as np
import rasterio
from scipy.interpolate import NearestNDInterpolator
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import glob
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import logging

# ============================
# SETUP
# ============================

# Setup logging
logging.basicConfig(filename=os.path.join(r"D:\Working_directory", "rf_script.log"), level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

csv_dir = r"D:\Path_to_CSV_folder"             #station csv files with columns for 'Date', 'Insitu" and 'SRP" rainfall values
coords_path = r"D:\Path_to_corrdinates_file\Coordinates.csv" #Coordinates file with 'Station name', 'Longitude' and 'Latitude' columns 
chirps_folder = r"D:\Path_to_SRP_raster_files"   #Folder containing SRPs raster files
output_folder = r"D:\OUTPUT_LS"
sample_raster_path = r"D:\Path_to_SRP_raster_files\P_CHIRPS.v2.0_mm-month-1_monthly_2012.01.01.tif"  #Sample raster file to use as reference for CRS, resolution, extent, etc.

os.makedirs(output_folder, exist_ok=True)
logging.info(f"Output directory created/verified: {output_folder}")

# ============================
# NSE FUNCTION
# ============================
def calculate_nse(observed, predicted):
    """Calculate Nash-Sutcliffe Efficiency between observed and predicted data."""
    observed = np.array(observed)
    predicted = np.array(predicted)
    mean_observed = np.mean(observed)
    numerator = np.sum((observed - predicted) ** 2)
    denominator = np.sum((observed - mean_observed) ** 2)
    if denominator == 0:
        logging.warning("NSE denominator is zero, returning nan")
        return np.nan
    return 1 - (numerator / denominator)

# ============================
# RF FUNCTION
# ============================
def rf_correction(observed, modeled, months):
    """Apply Random Forest Regression to correct modeled data."""
    # Remove NaN pairs
    valid_mask = ~(np.isnan(observed) | np.isnan(modeled) | np.isnan(months))
    if np.sum(valid_mask) < 3:
        logging.warning("Insufficient valid data (<3 points), returning original modeled data")
        return modeled, 1.0
    
    X = np.column_stack((modeled[valid_mask], months[valid_mask]))
    y = observed[valid_mask]
    
    # Train Random Forest
    rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
    try:
        rf.fit(X, y)
        # Predict for all data (including invalid, which will be NaN)
        X_all = np.column_stack((modeled, months))
        corrected = np.full_like(modeled, np.nan)
        corrected[valid_mask] = rf.predict(X[valid_mask])
        corrected = np.maximum(corrected, 0)  # Clip negative values
        
        # Compute correction factor
        correction_factor = np.median(corrected[valid_mask] / modeled[valid_mask][modeled[valid_mask] > 0]) if np.sum(modeled[valid_mask] > 0) > 0 else 1.0
        if not np.isfinite(correction_factor):
            logging.warning("Invalid correction factor, setting to 1.0")
            correction_factor = 1.0
    except Exception as e:
        logging.error(f"Random Forest fitting failed: {e}")
        return modeled, 1.0
    
    return corrected, correction_factor

# ============================
# STEP 1: LOAD DATA
# ============================
csv_files = glob.glob(os.path.join(csv_dir, "*.csv"))
station_data = {}
for file in csv_files:
    try:
        df = pd.read_csv(file, parse_dates=["Date"])
        df = df[(df["InSitu"] >= 0) & (df["WaPOR"] >= 0) | df["InSitu"].isna() | df["WaPOR"].isna()]
        station_name = os.path.basename(file).replace(".csv", "").strip("_")
        station_data[station_name] = df
        logging.info(f"Loaded data for station: {station_name}, rows: {len(df)}")
    except Exception as e:
        logging.error(f"Error loading {file}: {e}")

# ============================
# STEP 2: APPLY QM PER MONTH
# ============================
monthly_cfs = {}
valid_months = {}
for station, df in station_data.items():
    df["Month"] = df["Date"].dt.month
    df["Corrected"] = np.nan
    monthly_cfs[station] = {}
    valid_months[station] = []
    for month in range(1, 13):
        df_month = df[df["Month"] == month]
        valid_mask = ~(df_month["InSitu"].isna() | df_month["WaPOR"].isna())
        df_valid = df_month[valid_mask]
        if len(df_valid) >= 3:
            try:
                corrected_values, correction_factor = rf_correction(
                    df_valid["InSitu"].values, df_valid["WaPOR"].values, df_valid["Month"].values
                )
                df.loc[df_month.index[valid_mask], "Corrected"] = corrected_values
                valid_months[station].append(month)
                monthly_cfs[station][month] = correction_factor
                logging.info(f"Applied RF for {station}, month {month}: factor={correction_factor:.3f}")
            except Exception as e:
                logging.error(f"Error applying RF for {station}, month {month}: {e}")
                df.loc[df_month.index, "Corrected"] = df_month["WaPOR"]
                monthly_cfs[station][month] = 1.0
        else:
            logging.warning(f"Insufficient valid data (<3 points) for {station}, month {month}")
            df.loc[df_month.index, "Corrected"] = df_month["WaPOR"]
            monthly_cfs[station][month] = 1.0
    # Save corrected DataFrame
    output_path = os.path.join(output_folder, f"{station}_corrected.csv")
    df.to_csv(output_path, index=False)
    print(f"Saved corrected CSV for {station} to {output_path}")
    logging.info(f"Saved corrected CSV for {station} to {output_path}")
    station_data[station] = df

# ============================
# SAVE FACTORS
# ============================
cfs_list = []
for station in monthly_cfs:
    for month in range(1, 13):
        cfs_list.append({
            "Station": station,
            "Month": month,
            "Correction_Factor": monthly_cfs[station][month]
        })
cfs_df = pd.DataFrame(cfs_list)
cfs_path = os.path.join(output_folder, "correction_factors.csv")
cfs_df.to_csv(cfs_path, index=False)
print(f"Saved correction factors to {cfs_path}")
logging.info(f"Saved correction factors to {cfs_path}")

# ============================
# STEP 3: VALIDATION
# ============================
metrics_list = []
for station, df in station_data.items():
    if "Corrected" not in df.columns:
        print(f"No corrected data for {station}, skipping validation.")
        logging.warning(f"No corrected data for {station}, skipping validation")
        continue
    valid_df = df[df["Month"].isin(valid_months[station])]
    valid_mask = ~(valid_df["InSitu"].isna() | valid_df["Corrected"].isna())
    valid_df = valid_df[valid_mask]
    if valid_df.empty or len(valid_df) < 3:
        print(f"No valid data for validation for {station}, skipping.")
        logging.warning(f"No valid data for validation for {station}")
        continue
    try:
        correlation, _ = pearsonr(valid_df["InSitu"], valid_df["Corrected"])
        rmse = np.sqrt(mean_squared_error(valid_df["InSitu"], valid_df["Corrected"]))
        nse = calculate_nse(valid_df["InSitu"], valid_df["Corrected"])
        print(f"Station: {station}, Correction: RandomForest, Correlation: {correlation:.3f}, RMSE: {rmse:.3f}, NSE: {nse:.3f}")
        logging.info(f"Validation for {station}: Correlation={correlation:.3f}, RMSE={rmse:.3f}, NSE={nse:.3f}")
        metrics_list.append({"Station": station, "Correction": "RandomForest", "Correlation": correlation, "RMSE": rmse, "NSE": nse})

        # Visualize
        plt.figure(figsize=(10, 4))
        plt.plot(valid_df["Date"], valid_df["InSitu"], label="Insitu", marker="o")
        plt.plot(valid_df["Date"], valid_df["WaPOR"], label="WaPOR (CHIRPS)", marker="x")
        plt.plot(valid_df["Date"], valid_df["Corrected"], label="Corrected (RF)", marker="^")
        plt.title(f"Monthly Rainfall Comparison: {station}", fontsize=16)
        plt.xlabel("Date", fontsize=14)
        plt.ylabel("Rainfall (mm)", fontsize=14)
        plt.legend()
        plt.grid(True)
        plt.tick_params(axis='both', which='major', labelsize=14)  # bigger tick numbers
        # (optional) make legend font bigger too:
        plt.legend(fontsize=12)
        plt.tight_layout()
        plot_path = os.path.join(output_folder, f"{station}_comparison_rf.png")
        plt.savefig(plot_path, dpi=300)
        plt.close()
        logging.info(f"Saved comparison plot for {station} to {plot_path}")
    except Exception as e:
        logging.error(f"Error validating {station}: {e}")
        print(f"Error validating {station}: {e}")

# Save validation metrics to CSV
metrics_df = pd.DataFrame(metrics_list)
metrics_path = os.path.join(output_folder, "validation_metrics.csv")
metrics_df.to_csv(metrics_path, index=False)
print(f"Saved validation metrics to {metrics_path}")
logging.info(f"Saved validation metrics to {metrics_path}")


# ============================
# STEP 4: IDW + SMOOTHING
# ============================
from scipy.ndimage import gaussian_filter

def idw_interpolation(x, y, z, xi, yi, power=2):
    """Inverse Distance Weighting (IDW) interpolation."""
    dist = np.sqrt((xi[:, None] - x[None, :])**2 + (yi[:, None] - y[None, :])**2)
    dist[dist == 0] = 1e-12  # avoid division by zero
    weights = 1 / dist**power
    weights /= np.sum(weights, axis=1)[:, None]
    zi = np.sum(weights * z[None, :], axis=1)
    return zi

# Create smooth monthly correction factor grids using IDW
try:
    with rasterio.open(sample_raster_path) as src:
        chirps_transform = src.transform
        chirps_crs = src.crs
        chirps_width = src.width
        chirps_height = src.height
    logging.info(f"Loaded sample raster: {sample_raster_path}")
except Exception as e:
    logging.error(f"Error loading sample raster {sample_raster_path}: {e}")
    print(f"Error loading sample raster: {e}")
    raise

# Read coordinates
coords_df = pd.read_csv(coords_path)
x = coords_df["Longitude"].values
y = coords_df["Latitude"].values

# Prepare grid
cols, rows = np.meshgrid(np.arange(chirps_width), np.arange(chirps_height))
xs, ys = rasterio.transform.xy(chirps_transform, rows, cols)
xs = np.array(xs).flatten()
ys = np.array(ys).flatten()
grid_points = np.column_stack((xs, ys))

monthly_correction_grids = {}

for month in range(1, 13):
    try:
        # Station correction factors for this month
        z = np.array([monthly_cfs[station].get(month, 1.0) for station in station_data.keys()])

        # --- Step 1: Apply IDW interpolation ---
        correction_grid_flat = idw_interpolation(x, y, z, xs, ys, power=2)
        correction_grid = correction_grid_flat.reshape((chirps_height, chirps_width))

        # --- Step 2: Apply optional Gaussian smoothing to reduce artifacts ---
        correction_grid_smoothed = gaussian_filter(correction_grid, sigma=2)

        # --- Step 3: Save smoothed correction grid ---
        output_path = os.path.join(output_folder, f"CorrectionGrid_Month_{month:02d}_rf_idw_smoothed.tif")
        with rasterio.open(
            output_path,
            "w",
            driver="GTiff",
            height=chirps_height,
            width=chirps_width,
            count=1,
            dtype=correction_grid_smoothed.dtype,
            crs=chirps_crs,
            transform=chirps_transform,
        ) as dst:
            dst.write(correction_grid_smoothed, 1)

        monthly_correction_grids[month] = correction_grid_smoothed
        print(f"Saved smoothed IDW correction grid for month {month}")
        logging.info(f"Saved smoothed IDW correction grid for month {month}")

    except Exception as e:
        logging.error(f"Error creating grid for month {month}: {e}")
        print(f"Error creating grid for month {month}: {e}")

# ============================
# STEP 5: APPLY TO SRP
# ============================

chirps_files = glob.glob(os.path.join(chirps_folder, "*.tif"))
for file_path in chirps_files:
    try:
        base_name = os.path.basename(file_path)
        date_str = base_name.split("_")[-1].replace(".tif", "")  # e.g., "2012.01.01"
        date = pd.to_datetime(date_str, format="%Y.%m.%d")
        month = date.month

        # Build new filename: insert _corrected before the date
        prefix = base_name.replace(f"_{date_str}.tif", "")
        filename = f"{prefix}_corrected_{date_str}.tif"
        output_path = os.path.join(output_folder, filename)

        with rasterio.open(file_path) as src:
            chirps_data = src.read(1)
            meta = src.meta.copy()
            correction_grid = monthly_correction_grids.get(month, np.ones_like(chirps_data))
            corrected_data = chirps_data * correction_grid
            if src.nodata is not None:
                corrected_data[chirps_data == src.nodata] = src.nodata

            with rasterio.open(output_path, "w", **meta) as dst:
                dst.write(corrected_data, 1)
            print(f"Saved corrected raster: {output_path}")
            logging.info(f"Saved corrected raster: {output_path}")
    except Exception as e:
        logging.error(f"Error processing raster {file_path}: {e}")
        print(f"Error processing raster {file_path}: {e}")

# ============================
# STEP 6: EXAMPLE VALIDATION AT A PIXEL USING THE CORRECTED OUTPUT RASTER
# ============================
try:
    # Get the first corrected output file
    corrected_files = sorted(glob.glob(os.path.join(output_folder, "*_corrected_*.tif")))
    if not corrected_files:
        raise FileNotFoundError("No corrected files found in output folder.")

    sample_corrected_path = corrected_files[0]
    with rasterio.open(sample_corrected_path) as src:
        corrected_data = src.read(1)
        nodata = src.nodata

        # Extract date from filename
        base_name = os.path.basename(sample_corrected_path)
        date_str = base_name.split("_")[-1].replace(".tif", "")  # e.g., "2012.01.01"
        date = pd.to_datetime(date_str, format="%Y.%m.%d")
        month = date.month

        # Also read the original corresponding file for comparison
        original_filename = base_name.replace("_corrected_", "_")  # Reconstruct original name
        original_path = os.path.join(chirps_folder, original_filename)

        if not os.path.exists(original_path):
            print(f"Warning: Original file not found: {original_path}")
            logging.warning(f"Original file not found: {original_path}")
            original_value = "N/A"
        else:
            with rasterio.open(original_path) as src_orig:
                original_data = src_orig.read(1)
                original_value = original_data[0, 0] if nodata is None or original_data[0, 0] != nodata else "NoData"

        corrected_value = corrected_data[0, 0] if nodata is None or corrected_data[0, 0] != nodata else "NoData"

        print(f"Validation using: {base_name}")
        print(f"Date: {date_str}, Month: {month}")
        print(f"Original (top-left pixel): {original_value}")
        print(f"Corrected (top-left pixel): {corrected_value}")
        logging.info(f"Pixel validation - File: {base_name}, Original: {original_value}, Corrected: {corrected_value}")

except Exception as e:
    logging.error(f"Error in pixel validation: {e}")
    print(f"Error in pixel validation: {e}")